# Is Fandango Still Inflating Ratings?

In October 2015, data journalist Walt Hickey published an [analysis](https://fivethirtyeight.com/features/fandango-movies-ratings/) showing strong evidence that Fandango's movie rating system was biased: the star rating displayed to users was almost always rounded **up** from the actual rating stored in the page's HTML, sometimes by a whole star. Fandango said the rounding was caused by a bug and promised to fix it.

In this project, we analyze more recent movie ratings data to determine whether there has been any change in Fandango's rating system after Hickey's analysis. We compare ratings for popular movies released in 2015 (collected by Hickey) with ratings for popular movies released in 2016 (collected after his analysis).

## Understanding the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

previous = pd.read_csv('fandango_score_comparison.csv')
after = pd.read_csv('movie_ratings_16_17.csv')

previous.head(3)

In [ ]:
after.head(3)

In [ ]:
fandango_previous = previous[['FILM', 'Fandango_Stars', 'Fandango_Ratingvalue', 'Fandango_votes', 'Fandango_Difference']].copy()
fandango_after = after[['movie', 'year', 'fandango']].copy()

print(fandango_previous.shape)
print(fandango_after.shape)
fandango_previous.head(3)

In [ ]:
fandango_after.head(3)

Our goal is to determine whether there has been any change in Fandango's rating system after Hickey's analysis. The population of interest is therefore made of all the movie ratings stored on Fandango's website, regardless of the release year.

We want to describe this population at two different moments in time — before and after Hickey's analysis — so we need two samples. Reading the READMEs of the two datasets, it looks like neither sample was collected randomly:

- Hickey's data ([README](https://github.com/fivethirtyeight/data/blob/master/fandango/README.md)) only includes movies that had tickets on sale in 2015, at least 30 fan ratings on Fandango, and ratings on Rotten Tomatoes, Metacritic and IMDb.
- The second dataset ([README](https://github.com/mircealex/Movie_ratings_2016_17/blob/master/README.md)) contains ratings for the most popular movies with a significant number of votes released in 2016 and 2017.

Because of this sampling, some movies have no chance at all of being included (for example, movies with fewer than 30 fan ratings), so neither sample is likely to be representative of the whole population.

## Changing the Goal of Our Analysis

Instead of collecting new data, we slightly change the goal of our analysis so that the samples we have become much more useful. The new goal is:

> Determine whether there's any difference between Fandango's ratings for **popular** movies released in **2015** and Fandango's ratings for **popular** movies released in **2016**.

This new goal is still a good proxy for the initial one, and it gives us two populations of interest:

1. All Fandango ratings for popular movies released in 2015.
2. All Fandango ratings for popular movies released in 2016.

We use Hickey's benchmark of 30 fan ratings to define a movie as "popular".

## Isolating the Samples We Need

In [ ]:
(fandango_previous['Fandango_votes'] < 30).sum()

Every movie in Hickey's dataset has at least 30 fan ratings, so all of them are popular. The second dataset doesn't include the number of fan ratings, so we can't check this directly. One way to check would be to randomly sample a few movies and look up their number of fan ratings on Fandango's website; here we rely on the dataset's README, which states that it contains the most popular movies with a significant number of votes.

In [ ]:
fandango_after.sample(10, random_state=1)

Next, we isolate only the movies released in 2015 and 2016. Hickey's dataset has no `year` column, but the year is part of the `FILM` string, e.g. `'Avengers: Age of Ultron (2015)'`.

In [ ]:
fandango_previous['Year'] = fandango_previous['FILM'].str[-5:-1]
fandango_previous['Year'].value_counts()

In [ ]:
fandango_2015 = fandango_previous[fandango_previous['Year'] == '2015'].copy()
fandango_2015['Year'].value_counts()

In [ ]:
fandango_after['year'].value_counts()

In [ ]:
fandango_2016 = fandango_after[fandango_after['year'] == 2016].copy()
fandango_2016['year'].value_counts()

## Comparing Distribution Shapes for 2015 and 2016

In [ ]:
plt.style.use('fivethirtyeight')

fandango_2015['Fandango_Stars'].plot.kde(label='2015', legend=True, figsize=(8, 5.5))
fandango_2016['fandango'].plot.kde(label='2016', legend=True)

plt.title("Comparing distribution shapes for Fandango's ratings\n(2015 vs 2016)", y=1.07)
plt.xlabel('Stars')
plt.xlim(0, 5)
plt.xticks([0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5.0])
plt.show()

Two aspects stand out:

- Both distributions are strongly left skewed.
- The 2016 distribution is slightly shifted to the left relative to the 2015 distribution.

The left skew suggests that popular movies on Fandango are mostly rated high or very high. Given that Fandango also sells tickets, these high ratings are a bit suspicious. The slight left shift of the 2016 distribution is the more interesting part for our goal: it suggests there was a difference between Fandango's ratings for popular movies in 2015 and in 2016, and that ratings in 2016 were slightly lower.

## Comparing Relative Frequencies

The two samples have different sizes, so we compare relative frequencies (percentages) rather than absolute frequencies.

In [ ]:
print('2015' + '\n' + '-' * 16)
fandango_2015['Fandango_Stars'].value_counts(normalize=True).sort_index() * 100

In [ ]:
print('2016' + '\n' + '-' * 16)
fandango_2016['fandango'].value_counts(normalize=True).sort_index() * 100

In 2016, very high ratings (4.5 and 5 stars) had significantly lower percentages than in 2015: less than 1% of the movies got a perfect 5-star rating in 2016, compared to roughly 7% in 2015, and the share of 4.5-star ratings also dropped substantially. The minimum rating was also lower in 2016 — 2.5 stars instead of 3 stars.

On the other hand, a higher percentage of movies got 3.5 or 4 stars in 2016, which makes the direction of the change less obvious from the tables alone. To get a clearer picture, we compute a few summary statistics.

## Determining the Direction of the Change

In [ ]:
mean_2015 = fandango_2015['Fandango_Stars'].mean()
mean_2016 = fandango_2016['fandango'].mean()

median_2015 = fandango_2015['Fandango_Stars'].median()
median_2016 = fandango_2016['fandango'].median()

mode_2015 = fandango_2015['Fandango_Stars'].mode()[0]
mode_2016 = fandango_2016['fandango'].mode()[0]

summary = pd.DataFrame()
summary['2015'] = [mean_2015, median_2015, mode_2015]
summary['2016'] = [mean_2016, median_2016, mode_2016]
summary.index = ['mean', 'median', 'mode']
summary

In [ ]:
plt.style.use('fivethirtyeight')
summary['2015'].plot.bar(color='#0066FF', align='center', label='2015', width=.25)
summary['2016'].plot.bar(color='#CC0000', align='edge', label='2016', width=.25, rot=0, figsize=(8, 5))

plt.title('Comparing summary statistics: 2015 vs 2016', y=1.07)
plt.ylim(0, 5.5)
plt.yticks(np.arange(0, 5.1, .5))
plt.ylabel('Stars')
plt.legend(framealpha=0, loc='upper center')
plt.show()

In [ ]:
(summary.loc['mean', '2015'] - summary.loc['mean', '2016']) / summary.loc['mean', '2015']

The mean rating was lower in 2016 by approximately 0.2 stars — a drop of almost 5% relative to 2015. The median is the same for both distributions (4 stars), while the mode dropped from 4.5 stars in 2015 to 4 stars in 2016. This confirms the direction of the change we saw on the kernel density plots: on average, popular movies released in 2016 were rated slightly lower than popular movies released in 2015.

## Conclusion

Our analysis showed that there's a slight difference between Fandango's ratings for popular movies in 2015 and Fandango's ratings for popular movies in 2016: ratings for popular movies released in 2016 were lower on average than ratings for popular movies released in 2015.

We can't be completely sure what caused the change, but the chances are very high that it was caused by Fandango fixing the biased rating system after Hickey's analysis.